In [ ]:
import openml
import pandas as pd
import os
from openai import OpenAI
import json
from tqdm import tqdm
import time
import json
from itertools import islice
from collections import Counter

In [ ]:
client = OpenAI(api_key="") # insert openai api key for tokens

In [ ]:
flows = openml.flows.list_flows(output_format='dataframe')

flows = flows[['id', 'name']]

### Download all flows

In [ ]:
flows.to_csv('all_flows.csv', index=False)

In [ ]:
flows = pd.read_csv('all_flows.csv')

In [ ]:
BATCH_SIZE = 50 # Batch flows
output_file = "flow_algorithm_mapping.json"
batches = [flows.iloc[i:i+BATCH_SIZE] for i in range(0, len(flows), BATCH_SIZE)]

In [ ]:
output_json = {}
json_path = "flow_algorithm_mapping.json"

if os.path.exists(json_path):
    with open(json_path, "r") as f:
        output_json = json.load(f)

In [ ]:
system_prompt = """You are a machine learning expert specializing in analyzing OpenML flows. Your task is to classify flows into target algorithm categories or mark them as non-target.

**Target Algorithm Categories:**
1. **Support Vector Machine** - Support vector-based algorithms
2. **Random Forest** - Random forest ensemble methods  
3. **Decision Tree** - Single decision tree algorithms
4. **XGBoost** - Extreme gradient boosting methods

**Output for target algorithms:** `<flow_id>, <target_algorithm>, Classification`
**Output for non-target algorithms:** `<flow_id>, NOT TARGET, Other`

---

### Classification Approach:

**1. Pattern Recognition:**
Look for algorithm names, abbreviations, and common implementations that clearly indicate one of the target algorithms.

**2. Meta-Learning Models:**
For ensemble or wrapper methods, identify the base learner. If the base learner is a target algorithm, classify accordingly.

**3. Pipeline Processing:**
For pipelines with preprocessing steps, identify the final predictive model. Classify based on that final model if it's a target algorithm.

**4. Name Normalization:**
Ignore common prefixes (library names), suffixes (version numbers), and implementation details to focus on the core algorithm.

---

### General Guidelines:

**Support Vector Machine indicators:**
- Contains "SVM", "Support Vector", or similar vector-based terminology
- Common implementations and variants across different libraries

**Random Forest indicators:**
- Contains "Random Forest", "RandomForest", or equivalent ensemble terminology
- Tree-based ensemble methods with randomization

**Decision Tree indicators:**
- Contains "Tree", "Decision", or classic tree algorithm names
- Single tree methods (not ensembles)

**XGBoost indicators:**
- Contains "XGBoost", "Extreme Gradient", or gradient boosting terminology
- Advanced boosting implementations

---

### Processing Rules:

1. **Direct Algorithm Match**: If flow name clearly indicates a target algorithm, classify it
2. **Base Learner Extraction**: For meta-models, extract and evaluate the base algorithm
3. **Pipeline Final Model**: For preprocessing pipelines, evaluate the final predictive component
4. **Conservative Classification**: Only classify as target if clearly identifiable, otherwise use NOT TARGET

---

### Output Format:
```
<flow_id>, <algorithm_name>, Classification
```
OR
```
<flow_id>, NOT TARGET, Other
```

**Requirements:** 
- Exactly 3 comma-separated values
- Use exact names: "Support Vector Machine", "Random Forest", "Decision Tree", "XGBoost"
- Target algorithms use "Classification" type
- Non-targets use "NOT TARGET, Other"
- No explanations or additional text"""

### First parse and store in json mappings/flow_algorithm_mapping.json

In [ ]:
for i, batch in enumerate(tqdm(batches[2275:], desc="Processing Batches")):  # Process all batches
    
    # Skip batch if all flow IDs are already processed
    if all(str(row["id"]) in output_json for _, row in batch.iterrows()):
        continue

    # Create fast lookup for flow names
    id_to_name = dict(zip(batch["id"].astype(str), batch["name"]))

    # Prepare user message
    flow_lines = "\n".join([f"{fid}, {name}" for fid, name in id_to_name.items()])
    user_prompt = f"Please process the following OpenML flows:\n\n{flow_lines}"

    try:
        response = client.chat.completions.create(
            # model="gpt-4",
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        raw_reply = response.choices[0].message.content.strip()

        # Parse and store response - Updated for 3-part output
        for line in raw_reply.split("\n"):
            parts = [x.strip() for x in line.split(",", maxsplit=2)]  # Changed from maxsplit=3 to maxsplit=2
            if len(parts) == 3:  # Changed from 4 to 3
                fid, algorithm, alg_type = parts  # Updated unpacking
                if fid in id_to_name:
                    # Store with flow_id as key (which becomes index when converted to DataFrame)
                    output_json[fid] = {
                        "name": id_to_name[fid],
                        "algorithm_type": algorithm,  # Updated to match new prompt format
                        "classification_type": alg_type
                    }

    except Exception as e:
        pass  # Optional: log or retry

    # Save after every batch
    with open(json_path, "w") as f:
        json.dump(output_json, f, indent=2)

    time.sleep(1.0)  # Rate limiting

print("Done.")

In [ ]:
# Path to original and filtered JSON files
input_path = "flow_algorithm_mapping.json"          # Change this to your original JSON file if needed
output_path = "filtered_flow_algorithm_mapping.json"

# Load original JSON
with open(input_path, "r") as f:
    output_json = json.load(f)

# Print initial count
initial_count = len(output_json)
print(f"Initial number of entries: {initial_count}")

# Filter out entries with "NOT TARGET" algorithm_type
filtered_json = {
    fid: details for fid, details in output_json.items()
    if details.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# Print filtered count
filtered_count = len(filtered_json)
print(f"Number of entries after filtering: {filtered_count}")

# Save filtered JSON
with open(output_path, "w") as f:
    json.dump(filtered_json, f, indent=2)

print(f"✅ Filtered JSON saved to: {output_path}")


### Cleaning up gpt results 
remove not target and then also making sure all entries are clean

In [ ]:
# Load filtered JSON
with open(output_path, "r") as f:
    filtered_json = json.load(f)

# Extract all algorithm_type values
algorithm_types = [entry.get("algorithm_type", "").strip() for entry in filtered_json.values()]

# Get unique types and their count
unique_types = set(algorithm_types)
print(f"Number of unique algorithm types: {len(unique_types)}")
print("Unique algorithm types:")
for alg in sorted(unique_types):
    print("-", alg)


### identifying bad eggs

In [ ]:
target_algorithms = {"1", "2", "3", "4", "AdaBoost"}

# Load the filtered JSON
with open(output_path, "r") as f:
    data = json.load(f)

# Filter the entries with algorithm_type in the target list
core_subset = {
    fid: entry for fid, entry in data.items()
    if entry.get("algorithm_type", "").strip() in target_algorithms
}

# Print results
print(f"Found {len(core_subset)} entries with algorithm_type in {target_algorithms}")

# Save to a new file
with open("core_algorithm_subset.json", "w") as f:
    json.dump(core_subset, f, indent=2)

print("Saved to core_algorithm_subset.json")

In [ ]:
# Load the bad eggs ("core_algorithm_subset.json") for re-checking
with open("core_algorithm_subset.json", "r") as f:
    bad_eggs = json.load(f)

# Convert to DataFrame for batching
bad_df = pd.DataFrame.from_dict(bad_eggs, orient="index").reset_index(names="id")
bad_df["id"] = bad_df["id"].astype(str)

# Batch
BATCH_SIZE = 5
bad_batches = [bad_df.iloc[i:i+BATCH_SIZE] for i in range(0, len(bad_df), BATCH_SIZE)]

# Path to corrected output
corrected_path = "corrected_bad_eggs.json"
corrected_json = {}

# Load existing corrections if rerunning
if os.path.exists(corrected_path):
    with open(corrected_path, "r") as f:
        corrected_json = json.load(f)

In [ ]:
for i, batch in enumerate(tqdm(bad_batches, desc="Reprocessing bad eggs")):
    if all(str(row["id"]) in corrected_json for _, row in batch.iterrows()):
        continue

    id_to_name = dict(zip(batch["id"], batch["name"]))
    flow_lines = "\n".join([f"{fid}, {name}" for fid, name in id_to_name.items()])
    user_prompt = f"Please process the following OpenML flows:\n\n{flow_lines}"

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        raw_reply = response.choices[0].message.content.strip()

        for line in raw_reply.split("\n"):
            parts = [x.strip() for x in line.split(",", maxsplit=2)]
            if len(parts) == 3:
                fid, algorithm, alg_type = parts
                if fid in id_to_name:
                    corrected_json[fid] = {
                        "name": id_to_name[fid],
                        "algorithm_type": algorithm,
                        "classification_type": alg_type
                    }

    except Exception as e:
        print(f"⚠️ Error in batch {i}: {e}")
        continue

    # Save after each batch
    with open(corrected_path, "w") as f:
        json.dump(corrected_json, f, indent=2)

    time.sleep(1.0)

print("Reclassification complete.")

In [ ]:
with open("corrected_bad_eggs.json", "r") as f:
    data = json.load(f)

# Filter out entries with algorithm_type == "NOT TARGET"
filtered_data = {
    fid: entry for fid, entry in data.items()
    if entry.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# Save to new file
output_path = "filtered_flow_algorithm_mapping_v2.json"
with open(output_path, "w") as f:
    json.dump(filtered_data, f, indent=2)

# Report
print(f"Removed {len(data) - len(filtered_data)} 'NOT TARGET' entries.")
print(f"Remaining entries: {len(filtered_data)} saved to {output_path}")

In [ ]:
import json

# === File paths ===
main_path = "flows/filtered_flow_algorithm_mapping.json"
corrections_path = "flows/corrected_bad_eggs.json"
output_path = "flows/filtered_flow_algorithm_mapping_v2.json"

# === Load main + corrections ===
with open(main_path, "r") as f:
    main_data = json.load(f)

with open(corrections_path, "r") as f:
    corrections = json.load(f)

# === Overwrite entries from corrections ===
main_data.update(corrections)

# === Remove "NOT TARGET" entries (case-insensitive match) ===
cleaned_data = {
    fid: entry for fid, entry in main_data.items()
    if entry.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# === Save final cleaned mapping ===
with open(output_path, "w") as f:
    json.dump(cleaned_data, f, indent=2)

# === Report ===
removed = len(main_data) - len(cleaned_data)
print(f"Saved merged and cleaned mapping to: {output_path}")
print(f"Removed {removed} 'NOT TARGET' entries.")
print(f"Final count: {len(cleaned_data)} entries.")
